# Analytic fixed boundary to free boundary (#1608)

This notebook uses the public VAFT and ODS APIs to compare the direct Green response fit with TokaMaker `get_vfixed()` for Solov’ev and Guazzotto–Freidberg targets. Each route proceeds through a fixed-current free solve, optional shape refinement, and a second free solve with final PF currents frozen. Run native cells on a host with OpenFUSIONToolkit installed. Set `VAFT_RUN_NATIVE=1` and `VAFT_1608_OUT` to a fresh output directory. The solver runs only when explicitly enabled.

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from omas import ODS
from vaft.code.tokamaker import TokaMakerConfig, fixed_to_free
from vaft.process.equilibrium import (
    solovev_example, solve_guazzotto_freidberg,
    guazzotto_freidberg_to_equilibrium, as_equilibrium,
)

solovev = solovev_example(resolution=65)
gf_model = solve_guazzotto_freidberg(
    'limited', inverse_aspect_ratio=.33, nu=.5, elongation=1.6, triangularity=.3)
guazzotto = guazzotto_freidberg_to_equilibrium(
    gf_model, major_radius=.4, toroidal_field=.1, resolution=65)
targets = {'Solov’ev': solovev, 'Guazzotto–Freidberg': guazzotto}
[(name, eq.ip, eq.r0, eq.bt0) for name, eq in targets.items()]

## Machine geometry

The twelve synthetic one-turn PF circuits make the inverse problem reproducible. The current bounds are illustrative test limits of ±200 kA per circuit, not VEST hardware ratings. The closed rectangular wall touches the limited target at its inboard point.

In [ ]:
machine = ODS(consistency_check=False)
for i, angle in enumerate(np.linspace(0, 2*np.pi, 12, endpoint=False)):
    machine[f'pf_active.coil.{i}.name'] = f'PF{i+1}'
    element = f'pf_active.coil.{i}.element.0'
    machine[f'{element}.turns_with_sign'] = 1.
    for key, value in {'r': .4+.36*np.cos(angle), 'z': .65*np.sin(angle),
                       'width': .02, 'height': .02}.items():
        machine[f'{element}.geometry.rectangle.{key}'] = value
bounds = {f'PF{i+1}': (-2e5, 2e5) for i in range(12)}

## Fit, free solve, refine, verify

The direct route computes its plasma self-field with exact ring Green responses. The native route obtains the external vacuum target from an independent fixed-boundary TokaMaker solve. Both feed their fitted PF currents to a native free solve. `verify_refinement=True` removes the native shape constraints after optimization and resolves with the final currents fixed, retaining the finite-element state.

In [ ]:
results = {}
if os.environ.get('VAFT_RUN_NATIVE') == '1':
    output = Path(os.environ['VAFT_1608_OUT']).expanduser().resolve()
    output.mkdir(parents=True, exist_ok=False)
    for family, target in targets.items():
        left = float(target.lcfs.r.min())
        wall = ([left, .65, .65, left], [-.5, -.5, .5, .5])
        for backend in ('direct', 'tokamaker_vfixed'):
            key = (family, backend)
            config = TokaMakerConfig(
                profile_mode='equilibrium', dx_plasma=.04, dx_vacuum=.08,
                dx_coil=.02, limiter=wall, lim_zmax=None,
                neck_limiter_zmax=None, nthreads=2)
            fit_options = {'regularization': 1e-5, 'current_bounds': bounds}
            if backend == 'direct':
                fit_options.update(method='flux_normal', boundary_samples=64)
            results[key] = fixed_to_free(
                target, machine, output/f'{backend}_{len(results)}',
                fit_backend=backend, config=config, fit_options=fit_options,
                refine_shape=True, verify_refinement=True)
else:
    print('Set VAFT_RUN_NATIVE=1 and VAFT_1608_OUT to run the native cases.')

In [ ]:
for (family, backend), result in results.items():
    verified = result.verified_comparison
    q = verified.get('globals', {}).get('q95_field_integral', {})
    lcfs = verified.get('lcfs', {})
    print(f'{family:23s} {backend:17s} fit={result.fit.status:24s} '
          f'initial={result.status:15s} refined={result.refinement_status:15s} '
          f'verified={result.verification_status:15s} '
          f'LCFS max={1000*lcfs.get("max_distance_m", float("nan")):.3f} mm '
          f'q95 difference={q.get("difference")}')
    print('  initial currents [A]:', dict(result.initial_currents_A))
    print('  final currents [A]:', dict(result.refined_currents_A))

## Geometry comparison

The plotted final contour comes from the unconstrained free solve. Native FE saddle diagnostics are recorded separately because a gridded g-file can miss a narrow X-point.

In [ ]:
if results:
    fig, axes = plt.subplots(2, 2, figsize=(10, 10), sharex=True, sharey=True)
    for ax, ((family, backend), result) in zip(axes.flat, results.items()):
        target = targets[family]
        ax.plot(target.lcfs.r, target.lcfs.z, label='analytic target')
        if result.verified_free and result.verified_free.ok:
            solved = as_equilibrium(result.verified_free.gfile, convention=7)
            ax.plot(solved.lcfs.r, solved.lcfs.z, label='PF-fixed free solve')
            ax.plot(*target.magnetic_axis, 'ko', markersize=3)
            ax.plot(*solved.magnetic_axis, 'rx', markersize=5)
        ax.set_title(f'{family}, {backend}')
        ax.set_aspect('equal')
        ax.set_xlabel('R [m]')
        ax.set_ylabel('Z [m]')
    axes[0, 0].legend()
    fig.tight_layout()
    plt.show()

## Interpretation

Compare boundary field residuals, shape, topology, axis, q95, Ip and the other recorded global quantities. Ill-conditioned coil responses can yield different current vectors for nearly identical fields. `fit.accepted` reports linear inverse acceptance; `verification_status` reports the separate PF-fixed nonlinear solve. The full topology and pedestal matrix is reproduced by `python -m vaft.validation.studies.fixed_free_1608.matrix` on a server.